# Matrix and Vector Products in NumPy

This lesson turns the **Matrix and vector products** section of the [current NumPy linear-algebra reference](https://numpy.org/doc/stable/reference/routines.linalg.html) into small, hand-calculated examples. It is written for a reader who already knows ordinary matrix multiplication.

**Version note:** the course uses NumPy 2.5.3 or newer, so it includes newer routines such as `np.vecdot`, `np.matvec`, `np.vecmat`, and their documented `np.linalg` variants.

## Learning goals

By the end, you will be able to:

1. Predict the shape and meaning of a product before running it.
2. Choose among dot, inner, outer, matrix, vector, tensor, Kronecker, and cross products.
3. Recognize when NumPy conjugates a complex vector and when it does not.
4. Translate a small hand calculation into the matching NumPy expression.

Read each explanation, then run the matching code cell directly below it. The notebook uses a learn → run rhythm throughout.

## What the NumPy reference lists

The table below covers every entry in NumPy's current **Matrix and vector products** group. Several entries are Array-API-compatible `np.linalg` versions of the same underlying idea; they are shown together rather than taught twice.

| Concept | Functions in the reference | Main idea |
| --- | --- | --- |
| General dot product | `np.dot` | Dimension-dependent legacy/general product |
| Product chain | `np.linalg.multi_dot` | Matrix chain with an efficient parenthesization |
| Conjugating flattened dot | `np.vdot` | Flatten, conjugate the first input, then sum products |
| Conjugating vector dot | `np.vecdot`, `np.linalg.vecdot` | Dot corresponding vectors along one axis |
| Inner product | `np.inner` | Contract the last axis of each input |
| Outer product | `np.outer`, `np.linalg.outer` | Every pairwise scalar product |
| Matrix product | `np.matmul`, `@`, `np.linalg.matmul` | Ordinary matrix multiplication, including stacks |
| Matrix-vector product | `np.matvec` | Apply a matrix or matrix stack to a vector or vector stack |
| Vector-matrix product | `np.vecmat` | Multiply a vector or vector stack on the left |
| Tensor contraction | `np.tensordot`, `np.linalg.tensordot` | Sum products over selected axes |
| Einstein summation | `np.einsum`, `np.einsum_path` | Describe a contraction with index labels; inspect its plan |
| Matrix power | `np.linalg.matrix_power` | Repeated matrix multiplication of a square matrix |
| Kronecker product | `np.kron` | Replace every scalar in one array by a scaled block |
| Cross product | `np.linalg.cross` | A perpendicular vector from two 3-component vectors |

# A quick foundation

## Shapes, indices, and one essential warning

A NumPy vector such as `np.array([a, b, c])` has shape `(3,)`; it is one-dimensional. A matrix with `m` rows and `n` columns has shape `(m, n)`. An **axis** is simply a numbered direction: axis `0` moves down rows and axis `-1` means the last direction.

For two matrices `A` of shape `(m, k)` and `B` of shape `(k, n)`, ordinary matrix multiplication creates a result of shape `(m, n)`.

For entry `(i, j)`, multiply matching entries in row `i` of `A` and column `j` of `B`, then add them:

`(A @ B)[i, j] = A[i, 0] * B[0, j] + A[i, 1] * B[1, j] + ... + A[i, k - 1] * B[k - 1, j]`

In contrast, `A * B` is **elementwise** multiplication: it multiplies entries in matching positions and is not a matrix product.

## One-time setup

Run the next two cells once before working through the examples. The first preserves the notebook's Rich output helper; the second imports NumPy and defines the small display helper used below.

In [16]:
from rich import print

In [17]:
import numpy as np

np.set_printoptions(suppress=True)


def show(name, value):
    """Print a value with its NumPy shape."""
    print(f"{name} | shape={np.asarray(value).shape}")
    print(value)
    print()


print(f"NumPy version: {np.__version__}")

NumPy version: 2.5.3

## 1. `np.dot`: a dimension-dependent dot product

For two real vectors `u = [1, 2, 3]` and `v = [4, 5, 6]`, multiply matching entries and add:

`u dot v = 1 * 4 + 2 * 5 + 3 * 6 = 32`

For two 2-D arrays, `np.dot(A, B)` performs ordinary matrix multiplication. For example:

```text
A = [[1, 2],      B = [[5, 6],      A @ B = [[19, 22],
     [3, 4]]           [7, 8]]             [43, 50]]
```

The upper-left entry is `1 * 5 + 2 * 7 = 19`. Calculate the other entries in the same row-by-column way.

For arrays with more dimensions, `dot` contracts the last axis of the first input with the second-to-last axis of the second input. That rule is useful but less intuitive, so prefer `@` or `np.matmul` when you mean ordinary matrix multiplication. `dot` does **not** conjugate complex inputs.

In [18]:
# 1) np.dot: vector dot product and 2-D matrix product
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])
show("np.dot(u, v)", np.dot(u, v))
assert np.dot(u, v) == 32

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
dot_matrix = np.dot(A, B)
show("np.dot(A, B)", dot_matrix)
np.testing.assert_array_equal(dot_matrix, A @ B)

np.dot(u, v) | shape=()

32

np.dot(A, B) | shape=(2, 2)

[[19 22]
 [43 50]]

## 2. `np.linalg.multi_dot`: a chain of matrix products

Matrix multiplication is associative: `(A @ B) @ C` and `A @ (B @ C)` give the same numerical result whenever both are defined. The amount of work can still differ.

Suppose `A` has shape `(2, 3)`, `B` has shape `(3, 2)`, and `C` has shape `(2, 4)`.

- `(A @ B) @ C` needs `2 * 3 * 2 + 2 * 2 * 4 = 28` scalar multiplications.
- `A @ (B @ C)` needs `3 * 2 * 4 + 2 * 3 * 4 = 48` scalar multiplications.

`np.linalg.multi_dot([A, B, C])` computes the same chain and chooses a low-cost parenthesization for you. It treats a first 1-D input as a row vector and a last 1-D input as a column vector.

In [19]:
# 2) np.linalg.multi_dot: same answer, potentially less work
chain_a = np.arange(1, 7).reshape(2, 3)
chain_b = np.arange(1, 7).reshape(3, 2)
chain_c = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])

multi = np.linalg.multi_dot([chain_a, chain_b, chain_c])
left_associated = (chain_a @ chain_b) @ chain_c
right_associated = chain_a @ (chain_b @ chain_c)
show("multi_dot([A, B, C])", multi)
np.testing.assert_array_equal(multi, left_associated)
np.testing.assert_array_equal(multi, right_associated)

left_cost = 2 * 3 * 2 + 2 * 2 * 4
right_cost = 3 * 2 * 4 + 2 * 3 * 4
print(
    f"Estimated scalar multiplications: (A @ B) @ C = {left_cost}; A @ (B @ C) = {right_cost}"
)

multi_dot([A, B, C]) | shape=(2, 4)

[[22 28 22 28]
 [49 64 49 64]]

Estimated scalar multiplications: (A @ B) @ C = 28; A @ (B @ C) = 48

## 3. Conjugating dot products: `np.vdot` and `np.vecdot`

Complex numbers use a second kind of dot product. For `z = [z1, z2]` and `w = [w1, w2]`, the Hermitian (conjugating) dot product is:

`conj(z1) * w1 + conj(z2) * w2`

For `z = [1 + i, 2 - i]` and `w = [3 - 2i, -1 + 4i]`:

- `np.dot(z, w)` uses `z[i] * w[i]` and gives `7 + 10i`.
- A conjugating dot product uses `conj(z[i]) * w[i]` and gives `-5 + 2i`.

`np.vdot(a, b)` **flattens both inputs first**, conjugates the first input, and returns one scalar. With equally shaped matrices, it is their Frobenius inner product.

`np.vecdot(a, b)` and `np.linalg.vecdot(a, b)` also conjugate the first input, but keep leading dimensions. For a stack of vectors, they calculate one dot product per corresponding vector along the last axis (or a chosen `axis`).

In [20]:
# 3) np.vdot and np.vecdot: complex-conjugating dot products
z = np.array([1 + 1j, 2 - 1j])
w = np.array([3 - 2j, -1 + 4j])

show("np.dot(z, w)  # no conjugation", np.dot(z, w))
show("np.vdot(z, w)  # flatten + conjugate z", np.vdot(z, w))
show("np.vecdot(z, w)", np.vecdot(z, w))
show("np.linalg.vecdot(z, w)", np.linalg.vecdot(z, w))
np.testing.assert_allclose(np.dot(z, w), 7 + 10j)
np.testing.assert_allclose(np.vdot(z, w), -5 + 2j)
np.testing.assert_allclose(np.vecdot(z, w), np.vdot(z, w))

left_matrix = np.array([[1, 2], [3, 4]])
right_matrix = np.array([[5, 6], [7, 8]])
show("np.vdot(left_matrix, right_matrix)", np.vdot(left_matrix, right_matrix))
assert np.vdot(left_matrix, right_matrix) == 70

vector_stack = np.array([[1, 2, 3], [4, 5, 6]])
weights = np.array([1, 0, -1])
show("np.vecdot(vector_stack, weights)", np.vecdot(vector_stack, weights))
np.testing.assert_array_equal(np.vecdot(vector_stack, weights), [-2, -2])

np.dot(z, w)  # no conjugation | shape=()

(7+10j)

np.vdot(z, w)  # flatten + conjugate z | shape=()

(-5+2j)

np.vecdot(z, w) | shape=()

...

(-5+2j)

np.vdot(left_matrix, right_matrix) | shape=()

70

np.vecdot(vector_stack, weights) | shape=(2,)

[-2 -2]

## 4. `np.inner` and the outer-product pair

`np.inner(u, v)` is the ordinary, non-conjugating vector dot product. For the real vectors above it is again `32`. With arrays of more than one dimension, it contracts the **last** axis of each input.

For example, use rows `[1, 2]`, `[3, 4]` in one matrix and rows `[5, 6]`, `[7, 8]` in another. `np.inner` makes every row-to-row dot product:

```text
[[[1, 2] dot [5, 6], [1, 2] dot [7, 8]],
 [[3, 4] dot [5, 6], [3, 4] dot [7, 8]]]

= [[17, 23],
   [39, 53]]
```

The outer product does the opposite: it preserves both vector directions. For `u = [1, 2, 3]` and `v = [4, 5, 6]`, each result entry is `u[i] * v[j]`:

```text
[[4,  5,  6],
 [8, 10, 12],
 [12, 15, 18]]
```

`np.outer` flattens its inputs before forming this table. `np.linalg.outer` is the stricter Array-API-compatible variant: it accepts one-dimensional numeric vectors only.

In [21]:
# 4) np.inner, np.outer, and np.linalg.outer
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])
show("np.inner(u, v)", np.inner(u, v))
show("np.outer(u, v)", np.outer(u, v))
np.testing.assert_array_equal(np.outer(u, v), np.linalg.outer(u, v))

row_matrix_1 = np.array([[1, 2], [3, 4]])
row_matrix_2 = np.array([[5, 6], [7, 8]])
inner_rows = np.inner(row_matrix_1, row_matrix_2)
show("np.inner(row_matrix_1, row_matrix_2)", inner_rows)
np.testing.assert_array_equal(inner_rows, [[17, 23], [39, 53]])

np.inner(u, v) | shape=()

32

np.outer(u, v) | shape=(3, 3)

[[ 4  5  6]
 [ 8 10 12]
 [12 15 18]]

np.inner(row_matrix_1, row_matrix_2) | shape=(2, 2)

[[17 23]
 [39 53]]

## 5. Matrix products: `@`, `np.matmul`, and `np.linalg.matmul`

For ordinary matrix multiplication in NumPy, write `A @ B`. It is equivalent to `np.matmul(A, B)`. `np.linalg.matmul(A, B)` is the Array-API-compatible counterpart listed in the reference.

The shape rule is:

`(m, k) @ (k, n) -> (m, n)`

With `A = [[1, 2], [3, 4]]` and `B = [[5, 6], [7, 8]]`, the upper-left result is `1 * 5 + 2 * 7 = 19`. Calculate the other entries the same way to obtain:

```text
[[19, 22],
 [43, 50]]
```

Unlike `vdot`, `vecdot`, and `vecmat`, matrix multiplication does **not** conjugate complex values. For stacks of matrices, `matmul` uses the final two axes as matrix axes and broadcasts the earlier axes.

In [22]:
# 5) Ordinary matrix products: @, np.matmul, and np.linalg.matmul
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
at_product = A @ B
show("A @ B", at_product)
show("np.matmul(A, B)", np.matmul(A, B))
show("np.linalg.matmul(A, B)", np.linalg.matmul(A, B))
np.testing.assert_array_equal(at_product, [[19, 22], [43, 50]])
np.testing.assert_array_equal(at_product, np.matmul(A, B))
np.testing.assert_array_equal(at_product, np.linalg.matmul(A, B))

matrix_stack = np.array([[[1, 0], [0, 1]], [[2, 0], [0, 2]]])
show("matrix_stack @ B (one B broadcast to both matrices)", matrix_stack @ B)

A @ B | shape=(2, 2)

[[19 22]
 [43 50]]

np.matmul(A, B) | shape=(2, 2)

[[19 22]
 [43 50]]

np.linalg.matmul(A, B) | shape=(2, 2)

[[19 22]
 [43 50]]

matrix_stack @ B (one B broadcast to both matrices) | shape=(2, 2, 2)

[[[ 5  6]
  [ 7  8]]

 [[10 12]
  [14 16]]]

## 6. Matrix-vector and vector-matrix products: `np.matvec` and `np.vecmat`

For `A = [[1, 2], [3, 4]]` and `v = [5, 6]`, matrix-vector multiplication makes one dot product per row:

```text
A @ v = [1 * 5 + 2 * 6,
         3 * 5 + 4 * 6]
      = [17, 39]
```

That is `np.matvec(A, v)`. It is especially clear for a stack of matrices and a stack of vectors, where NumPy computes a matrix-vector product for each broadcasted pair.

For the vector on the left:

```text
v @ A = [5 * 1 + 6 * 3,
         5 * 2 + 6 * 4]
      = [23, 34]
```

For real inputs, `np.vecmat(v, A)` gives this result. For complex inputs, it uses `sum(conj(v[i]) * A[i, j])`, so it conjugates the vector. Use `v @ A` when you explicitly want the ordinary, non-conjugating left matrix product.

In [23]:
# 6) np.matvec and np.vecmat
M = np.array([[1, 2], [3, 4]])
real_vector = np.array([5, 6])
matrix_vector = np.matvec(M, real_vector)
vector_matrix = np.vecmat(real_vector, M)
show("np.matvec(M, [5, 6])", matrix_vector)
show("np.vecmat([5, 6], M)", vector_matrix)
np.testing.assert_array_equal(matrix_vector, M @ real_vector)
np.testing.assert_array_equal(vector_matrix, real_vector @ M)

matrices = np.array([[[1, 0], [0, 1]], [[2, 0], [0, 2]]])
vectors = np.array([[1, 2], [3, 4]])
show("np.matvec(stacked matrices, stacked vectors)", np.matvec(matrices, vectors))

complex_vector = np.array([1 + 2j, 3 - 1j])
identity = np.eye(2)
show("complex_vector @ identity (no conjugation)", complex_vector @ identity)
show("np.vecmat(complex_vector, identity)", np.vecmat(complex_vector, identity))
np.testing.assert_allclose(
    np.vecmat(complex_vector, identity), np.conjugate(complex_vector)
)

np.matvec(M, [5, 6]) | shape=(2,)

[17 39]

np.vecmat([5, 6], M) | shape=(2,)

[23 34]

np.matvec(stacked matrices, stacked vectors) | shape=(2, 2)

[[1 2]
 [6 8]]

complex_vector @ identity (no conjugation) | shape=(2,)

[1.+2.j 3.-1.j]

np.vecmat(complex_vector, identity) | shape=(2,)

[1.-2.j 3.+1.j]

## 7. Tensor contraction: `np.tensordot` and `np.linalg.tensordot`

A tensor is just an array with any number of axes. `tensordot` says exactly which axes to multiply and sum, called **contracting** axes. The output keeps all uncontracted axes of the first input, followed by all uncontracted axes of the second.

For two `(2, 2)` matrices `A` and `B`:

- `axes=1` contracts A's last axis with B's first axis. This is ordinary matrix multiplication: `C[i, j] = sum(A[i, r] * B[r, j])`.
- `axes=0` contracts nothing. It keeps every pair of entries, so `T[i, j, p, q] = A[i, j] * B[p, q]` and has shape `(2, 2, 2, 2)`. For example, `T[0, 1, 1, 0] = A[0, 1] * B[1, 0]`.

For a nonstandard pairing, pass two axis lists such as `axes=([1, 2], [0, 1])`. The paired axis sizes must match. `np.linalg.tensordot` is the documented Array-API-compatible version.

In [24]:
# 7) np.tensordot and np.linalg.tensordot
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
contract_one_axis = np.tensordot(A, B, axes=1)
show("np.tensordot(A, B, axes=1)", contract_one_axis)
np.testing.assert_array_equal(contract_one_axis, A @ B)
np.testing.assert_array_equal(contract_one_axis, np.linalg.tensordot(A, B, axes=1))

contract_no_axes = np.tensordot(A, B, axes=0)
print(f"np.tensordot(A, B, axes=0).shape = {contract_no_axes.shape}")
print(f"Entry [0, 1, 1, 0] = {contract_no_axes[0, 1, 1, 0]} = A[0, 1] * B[1, 0]")
assert contract_no_axes[0, 1, 1, 0] == A[0, 1] * B[1, 0]

np.tensordot(A, B, axes=1) | shape=(2, 2)

[[19 22]
 [43 50]]

np.tensordot(A, B, axes=0).shape = (2, 2, 2, 2)

Entry [0, 1, 1, 0] = 14 = A[0, 1] * B[1, 0]

## 8. Einstein summation: `np.einsum` and `np.einsum_path`

`einsum` writes the axes of a calculation as index labels. A label repeated across inputs is summed; a label that remains after `->` appears in the output. This is a compact way to describe many earlier products:

| Expression | Read it as | Equivalent idea |
| --- | --- | --- |
| `'i,i->'` | sum over `i` | vector dot product |
| `'i,j->ij'` | retain `i` and `j` | outer product |
| `'ij,j->i'` | sum the shared `j` | matrix-vector product |
| `'ij,jk->ik'` | sum the shared `j` | matrix product |

For the usual matrix product, `'ij,jk->ik'` means `C[i, k] = sum(A[i, j] * B[j, k])`. In the small example above, `C[0, 0] = 1 * 5 + 2 * 7 = 19`.

`np.einsum_path` does not calculate the final product. It reports a low-cost order in which `einsum` can contract three or more operands, much like `multi_dot` chooses parentheses for a matrix chain.

In [25]:
# 8) np.einsum: index labels describe a product
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])

einsum_dot = np.einsum("i,i->", u, v)
einsum_outer = np.einsum("i,j->ij", u, v)
einsum_matrix = np.einsum("ij,jk->ik", A, B)
show("np.einsum('i,i->', u, v)", einsum_dot)
show("np.einsum('i,j->ij', u, v)", einsum_outer)
show("np.einsum('ij,jk->ik', A, B)", einsum_matrix)
assert einsum_dot == np.dot(u, v)
np.testing.assert_array_equal(einsum_outer, np.outer(u, v))
np.testing.assert_array_equal(einsum_matrix, A @ B)

np.einsum('i,i->', u, v) | shape=()

32

np.einsum('i,j->ij', u, v) | shape=(3, 3)

[[ 4  5  6]
 [ 8 10 12]
 [12 15 18]]

np.einsum('ij,jk->ik', A, B) | shape=(2, 2)

[[19 22]
 [43 50]]

In [26]:
# 8b) np.einsum_path: plan a three-matrix contraction
chain_a = np.arange(1, 7).reshape(2, 3)
chain_b = np.arange(1, 7).reshape(3, 2)
chain_c = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])
path, report = np.einsum_path(
    "ij,jk,kl->il", chain_a, chain_b, chain_c, optimize="greedy"
)
print("Suggested contraction path:", path)
print("\n".join(report.splitlines()[:10]))
planned_product = np.einsum("ij,jk,kl->il", chain_a, chain_b, chain_c, optimize=path)
np.testing.assert_array_equal(
    planned_product, np.linalg.multi_dot([chain_a, chain_b, chain_c])
)

Suggested contraction path:
['einsum_path', (0, 1), (0, 1)]

Complete contraction:  ij,jk,kl->il
         Naive scaling:  4
     Optimized scaling:  3
      Naive FLOP count:  1.440e+02
  Optimized FLOP count:  5.700e+01
   Theoretical speedup:  2.526
  Largest intermediate:  8.000e+00 elements
--------------------------------------------------------------------------
scaling                  current                                remaining
--------------------------------------------------------------------------

## 9. `np.linalg.matrix_power`: repeated matrix multiplication

For a square matrix, matrix power is not elementwise power. Let `P = [[1, 2], [3, 4]]`.

```text
P @ P = [[1 * 1 + 2 * 3, 1 * 2 + 2 * 4],
         [3 * 1 + 4 * 3, 3 * 2 + 4 * 4]]
      = [[7, 10],
         [15, 22]]
```

So `np.linalg.matrix_power(P, 2)` gives `[[7, 10], [15, 22]]`.

By contrast, `P ** 2` squares each entry separately and gives `[[1, 4], [9, 16]]`. `np.linalg.matrix_power(P, 0)` returns the identity matrix, and a negative power first uses the inverse when it exists.

In [27]:
# 9) np.linalg.matrix_power: repeated matrix multiplication
P = np.array([[1, 2], [3, 4]])
show("P @ P", P @ P)
show("np.linalg.matrix_power(P, 2)", np.linalg.matrix_power(P, 2))
show("P ** 2  # elementwise, not a matrix power", P**2)
show("np.linalg.matrix_power(P, 0)", np.linalg.matrix_power(P, 0))
show("np.linalg.matrix_power(P, -1)", np.linalg.matrix_power(P, -1))
np.testing.assert_array_equal(np.linalg.matrix_power(P, 2), P @ P)
np.testing.assert_array_equal(np.linalg.matrix_power(P, 0), np.eye(2, dtype=int))
np.testing.assert_allclose(np.linalg.matrix_power(P, -1), np.linalg.inv(P))

P @ P | shape=(2, 2)

[[ 7 10]
 [15 22]]

np.linalg.matrix_power(P, 2) | shape=(2, 2)

[[ 7 10]
 [15 22]]

P ** 2  # elementwise, not a matrix power | shape=(2, 2)

[[ 1  4]
 [ 9 16]]

np.linalg.matrix_power(P, 0) | shape=(2, 2)

[[1 0]
 [0 1]]

np.linalg.matrix_power(P, -1) | shape=(2, 2)

[[-2.   1. ]
 [ 1.5 -0.5]]

## 10. `np.kron`: the Kronecker product

The Kronecker product makes a block matrix. Let:

```text
A = [[1, 2],      B = [[0, 5],
     [3, 4]]           [6, 7]]
```

Replace every entry `A[i, j]` with the entire block `A[i, j] * B`:

```text
np.kron(A, B) = [[1 * B, 2 * B],
                 [3 * B, 4 * B]]

              = [[ 0,  5,  0, 10],
                 [ 6,  7, 12, 14],
                 [ 0, 15,  0, 20],
                 [18, 21, 24, 28]]
```

If `A` has shape `(r, c)` and `B` has shape `(s, t)`, `np.kron(A, B)` has shape `(r * s, c * t)`.

In [28]:
# 10) np.kron: scaled blocks
A = np.array([[1, 2], [3, 4]])
B = np.array([[0, 5], [6, 7]])
kronecker = np.kron(A, B)
show("np.kron(A, B)", kronecker)
expected_kron = np.array(
    [[0, 5, 0, 10], [6, 7, 12, 14], [0, 15, 0, 20], [18, 21, 24, 28]]
)
np.testing.assert_array_equal(kronecker, expected_kron)
print("Top-left 2-by-2 block equals 1 * B:")
print(kronecker[:2, :2])

np.kron(A, B) | shape=(4, 4)

[[ 0  5  0 10]
 [ 6  7 12 14]
 [ 0 15  0 20]
 [18 21 24 28]]

Top-left 2-by-2 block equals 1 * B:

[[0 5]
 [6 7]]

## 11. `np.linalg.cross`: a 3-D vector product

For `a = [a1, a2, a3]` and `b = [b1, b2, b3]`, calculate the cross product component by component:

```text
cross(a, b) = [a2 * b3 - a3 * b2,
               a3 * b1 - a1 * b3,
               a1 * b2 - a2 * b1]
```

With `a = [1, 2, 3]` and `b = [4, 5, 6]`:

```text
cross(a, b) = [2 * 6 - 3 * 5,
               3 * 4 - 1 * 6,
               1 * 5 - 2 * 4]
            = [-3, 6, -3]
```

The result is perpendicular to both inputs: its dot product with each is zero. `np.linalg.cross` requires 3-element vectors and can calculate many pairwise cross products in a stack.

In [29]:
# 11) np.linalg.cross: a perpendicular 3-D vector
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
cross_product = np.linalg.cross(a, b)
show("np.linalg.cross(a, b)", cross_product)
np.testing.assert_array_equal(cross_product, [-3, 6, -3])
assert np.dot(cross_product, a) == 0
assert np.dot(cross_product, b) == 0
print("The two zero dot products confirm perpendicularity.")

np.linalg.cross(a, b) | shape=(3,)

[-3  6 -3]

The two zero dot products confirm perpendicularity.

## Quick product chooser

| If you want to… | Prefer | Watch out for |
| --- | --- | --- |
| Multiply two ordinary matrices | `A @ B` | Inner dimensions must agree |
| Multiply a chain of matrices efficiently | `np.linalg.multi_dot([...])` | It is for a chain, not elementwise work |
| Find a real-vector similarity/projection | `np.dot(u, v)` or `np.inner(u, v)` | Neither conjugates complex values |
| Find a complex inner product | `np.vdot` or `np.vecdot` | `vdot` flattens; `vecdot` preserves batches |
| Make all pairwise vector products | `np.outer(u, v)` | It produces a matrix, not a scalar |
| Apply many matrices to many vectors | `np.matvec` | Read the last two/last axes as matrix/vector axes |
| Specify exactly which axes are summed | `np.tensordot` or `np.einsum` | Write down the axes or labels first |
| Repeat a matrix product | `np.linalg.matrix_power` | Do not use `**` |
| Build a scaled block matrix | `np.kron` | Output size multiplies along each axis |
| Form a perpendicular 3-D vector | `np.linalg.cross` | Inputs must have exactly three components |

## Short practice — predict before you run

1. Calculate `[2, -1] dot [4, 3]` by hand.
2. Compute `[[2, 1], [0, -1]] @ [3, 4]` by taking one dot product per row.
3. Predict the shape and block layout of `np.kron(np.eye(2), [[2, 3], [4, 5]])`.
4. Explain why `np.vdot([[1, 2], [3, 4]], [[5, 6], [7, 8]])` is a scalar, while `np.inner` on those matrices is a 2-by-2 array.

Run the next cell only after attempting the first three questions.

In [30]:
# Practice answers and a compact final check
practice_dot = np.dot([2, -1], [4, 3])
practice_matrix = np.array([[2, 1], [0, -1]])
practice_vector = np.array([3, 4])
practice_matvec = np.matvec(practice_matrix, practice_vector)
practice_kron = np.kron(np.eye(2, dtype=int), np.array([[2, 3], [4, 5]]))

show("[2, -1] dot [4, 3]", practice_dot)
show("[[2, 1], [0, -1]] @ [3, 4]", practice_matvec)
show("kron(I_2, [[2, 3], [4, 5]])", practice_kron)

assert practice_dot == 5
np.testing.assert_array_equal(practice_matvec, [10, -4])
np.testing.assert_array_equal(
    practice_kron, [[2, 3, 0, 0], [4, 5, 0, 0], [0, 0, 2, 3], [0, 0, 4, 5]]
)
print("All product examples and practice checks passed.")

[2, -1] dot [4, 3] | shape=()

5

[[2, 1], [0, -1]] @ [3, 4] | shape=(2,)

[10 -4]

kron(I_2, [[2, 3], [4, 5]]) | shape=(4, 4)

[[2 3 0 0]
 [4 5 0 0]
 [0 0 2 3]
 [0 0 4 5]]

All product examples and practice checks passed.